# Recall@5-optimal trace ranking — CheckThat! 2026 Task 2 (EN / AR / ES)

**Method.** A trace is relevant iff its verdict equals the gold label, so expected Recall@5 is
maximised exactly by ranking *verdict groups* by `p(label = c) / n_c`. `p` comes from a logistic
regression over verdict counts, exact-claim memory, claim embeddings, per-verdict-group trace
embeddings and k-NN label votes (blocks chosen by 5-fold CV on train+val), with covariate-shift
weights (train+val vs. unlabeled test).

**Order of stages (results are saved after every stage)**
| Stage | Device | Time | Output |
|---|---|---|---|
| A. rank AR, ES (8B embeddings) and EN (counts + memory) | CPU | ~20 min | `results_A/` |
| B. English per-trace embeddings: preflight, then shards | GPU | 0.6B ≈ 0.5–1 h, 4B ≈ 3–5 h, 8B-4bit ≈ 7–10 h | `en_trace_emb/` |
| C. rank EN with its new embeddings | CPU | ~10 min | `results_C/` |

**Settings:** Accelerator = *GPU T4 x2*, Internet = *On*. Attach the CLEF data and the
dataset(s) holding `train/val/test_qwen3emb8b_ar_v1.npz` for Arabic and Spanish.
Use **Save Version → Save & Run All** so the run survives closing the browser; every log line
appears under its cell.

## 1. Configuration

In [ ]:
import glob, os, subprocess, sys

DATA_ROOT = "/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data"

# Folders with {train,val,test}_qwen3emb8b_ar_v1.npz. None = auto-detect (path mentioning "arab"/"span").
AR_EMB_DIR = None
ES_EMB_DIR = None

# English embeddings. Options (2x T4, ~45M tokens):
#   "Qwen/Qwen3-Embedding-0.6B", EN_QUANT=None    ~0.5-1 h   (quick check whether trace embeddings help)
#   "Qwen/Qwen3-Embedding-4B",   EN_QUANT=None    ~3-5 h
#   "Qwen/Qwen3-Embedding-8B",   EN_QUANT="4bit"  ~7-10 h, may need 2 sessions (RESUME_ROOT); one nf4
#                                                  copy (~5.5 GB) per T4. 4-bit saves memory, not compute.
# The preflight line in Stage B prints the real ETA before the long run starts.
EN_EMB_MODEL  = "Qwen/Qwen3-Embedding-8B"
EN_QUANT      = "4bit"    # None or "4bit"
EN_MAX_LEN    = 512       # tokens per trace (median trace ~250 tokens)
EN_EMB_DIR    = "/kaggle/working/en_trace_emb"
RESUME_ROOT   = None      # "/kaggle/input/<saved output of an unfinished run>" to reuse its shards
TIME_BUDGET_H = 9.0       # embedding stops starting new shards after this; leaves time for Stage C

WORK = "/kaggle/working"


def find_npz_dir(word):
    hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/test_qwen3emb8b_ar_v1.npz", recursive=True)]
    hits = [h for h in hits if word in h.lower()] or hits
    return hits[0] if hits else None

AR_EMB_DIR = AR_EMB_DIR or find_npz_dir("arab")
ES_EMB_DIR = ES_EMB_DIR or find_npz_dir("span")
print("DATA_ROOT :", DATA_ROOT, "| exists:", os.path.exists(DATA_ROOT))
print("AR_EMB_DIR:", AR_EMB_DIR)
print("ES_EMB_DIR:", ES_EMB_DIR)
assert AR_EMB_DIR is None or AR_EMB_DIR != ES_EMB_DIR, "AR and ES point to the same folder - set them explicitly"


def run_live(cmd):
    """Runs a command and streams its output into this cell (plain subprocess.run hides it)."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"command failed with exit code {p.returncode}")

## 2. Environment check

In [ ]:
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} {p.total_memory / 2**30:.1f} GiB")
from packaging import version
if version.parse(transformers.__version__) < version.parse("4.51"):   # Qwen3 needs >= 4.51
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.51"], check=True)
    raise SystemExit("transformers upgraded -> Run > Restart & run all")
if EN_QUANT == "4bit":
    try:
        import bitsandbytes
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes>=0.43"], check=True)
        import bitsandbytes
    print("bitsandbytes", bitsandbytes.__version__)

## 3. Scripts

In [ ]:
%%writefile /kaggle/working/embed_traces.py
"""
Per-trace Qwen3 embeddings for the ENGLISH split (Arabic / Spanish already exist in this format).

Output (same layout as the existing AR/ES files, so rank_traces.py reads it unchanged):
    {split}_trace_emb_en.npz : key "id_{i}" -> float16 (1 + #non-empty traces, dim)
        row 0 = claim, rows 1.. = non-empty reasoning traces in original order, L2-normalised
        i = position for train/val, query_id for test

Runs inside the notebook kernel (one thread + one model copy per GPU), so every log line is
visible in the cell. Steps:
  1. preflight : embeds ~100 real texts, prints tokens/s, NaN count, GPU memory and the ETA for
                 the whole job. fp16 NaNs -> automatic fp32 retry if the model fits, else abort.
  2. shards    : 200 claims per shard, each saved to OUT_DIR/shards as soon as it is done;
                 finished shards (also from --resume-root) are skipped.
  3. merge     : once every shard exists, writes the three final npz files.

Memory: weights stream straight to the GPU (low_cpu_mem_usage), batches are capped by a
padded-token budget, and a CUDA OOM halves the batch instead of crashing.
Models that do not fit one GPU (8B fp16 = 15 GB on a 15 GB T4) are split over all GPUs
(single copy, slower) -- or pass quant="4bit" (bitsandbytes nf4): 8B then needs ~5.5 GB, so each
T4 holds its own copy. 4-bit saves memory, not compute: 8B-4bit is ~2x slower than 4B fp16.

Notebook:   import embed_traces; embed_traces.run(data_root=..., out_dir=..., model="Qwen/Qwen3-Embedding-0.6B")
CLI:        python embed_traces.py --data-root ... --out-dir ... --model ...
"""

import argparse
import glob
import json
import os
import queue
import re
import shutil
import threading
import time

import numpy as np

SPLITS = {"train": "train.json", "val": "validation.json", "test": "test.json"}


def log(msg):
    print(time.strftime("%H:%M:%S"), msg, flush=True)


def find(root, name):
    hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True), key=len)
    hits = [h for h in hits if "arabic" not in h.lower() and "spanish" not in h.lower()]
    if not hits:
        raise FileNotFoundError(f"cannot find {name} under {root}")
    return hits[0]


def load_json(path):
    with open(path, encoding="utf-8") as f:
        s = f.read()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        return json.loads(re.sub(r'\{\s*\nx?(\s+)"(?![A-Za-z_]+"\s*:)', r'{\n        "Reasoning_traces": [\n\1"', s))


def claim_texts(e):
    return [str(e["claim"])] + [str(t) for t in e["Reasoning_traces"] if str(t).strip()]


def shard_path(out_dir, split, start):
    return os.path.join(out_dir, "shards", f"{split}_{start:06d}.npz")


# ============================================================
# MODEL REPLICA
# ============================================================
class Encoder:
    def __init__(self, model_id, dtype, devices, max_len, gpu_mem, quant=None):
        import torch
        from transformers import AutoModel, AutoTokenizer

        self.torch = torch
        self.max_len = max_len
        self.tok = AutoTokenizer.from_pretrained(model_id, padding_side="left")
        kw = dict(torch_dtype=dtype, low_cpu_mem_usage=True, attn_implementation="sdpa")
        if quant == "4bit":
            from transformers import BitsAndBytesConfig
            # nf4 weights; `dtype` is the compute precision (fp16, or fp32 after a NaN fallback)
            kw["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True)
        if len(devices) > 1:  # model parallel: one copy split over several GPUs
            kw["device_map"] = "auto"
            kw["max_memory"] = {d: gpu_mem for d in devices}
        else:
            kw["device_map"] = {"": devices[0]}
        self.model = AutoModel.from_pretrained(model_id, **kw).eval()
        self.dev = next(self.model.parameters()).device

    def encode(self, batch):
        torch = self.torch
        with torch.inference_mode():
            enc = self.tok(batch, padding=True, truncation=True, max_length=self.max_len,
                           return_tensors="pt").to(self.dev)
            h = self.model(**enc).last_hidden_state[:, -1].float()   # last-token pooling (left padding)
            h = torch.nn.functional.normalize(h, dim=-1)
            return h.cpu().numpy(), int(enc["input_ids"].numel())

    def encode_safe(self, batch):
        """Returns (embeddings, padded_tokens, n_bad). OOM -> split the batch; NaN rows -> zeros."""
        try:
            out, ntok = self.encode(batch)
        except self.torch.cuda.OutOfMemoryError:
            self.torch.cuda.empty_cache()
            if len(batch) == 1:
                raise
            mid = len(batch) // 2
            a, ta, ba = self.encode_safe(batch[:mid])
            b, tb, bb = self.encode_safe(batch[mid:])
            return np.vstack([a, b]), ta + tb, ba + bb
        bad = ~np.isfinite(out).all(1)
        out[bad] = 0.0
        return out, ntok, int(bad.sum())


def embed_texts(enc, texts, token_budget, max_batch):
    """Length-sorted, token-budget batching. Returns (float16 array, padded tokens, #NaN rows)."""
    lens = np.array([len(t) for t in texts])
    order = np.argsort(-lens)                 # longest first: an OOM shows up immediately
    embs = [None] * len(texts)
    ntok = nbad = 0
    i = 0
    while i < len(order):
        longest = min(enc.max_len, int(lens[order[i]] / 3.5) + 8)   # chars/3.5 over-estimates tokens
        bs = max(1, min(max_batch, token_budget // longest))
        idx = order[i : i + bs]
        out, t, b = enc.encode_safe([texts[j] for j in idx])
        for j, v in zip(idx, out):
            embs[j] = v.astype(np.float16)
        ntok += t
        nbad += b
        i += bs
    return np.stack(embs), ntok, nbad


# ============================================================
# MAIN
# ============================================================
def run(data_root, out_dir, model="Qwen/Qwen3-Embedding-0.6B", max_len=512, shard_size=200,
        token_budget=16384, max_batch=64, time_budget_h=9.0, resume_root=None, gpu_mem="13GiB",
        preflight_only=False, quant=None):
    import torch

    t_start = time.time()
    os.makedirs(os.path.join(out_dir, "shards"), exist_ok=True)
    if resume_root:
        n = 0
        for f in glob.glob(os.path.join(resume_root, "**", "shards", "*.npz"), recursive=True):
            dst = os.path.join(out_dir, "shards", os.path.basename(f))
            if ".tmp" not in f and not os.path.exists(dst):
                shutil.copy(f, dst)
                n += 1
        log(f"resumed {n} finished shards from {resume_root}")

    data = {s: load_json(find(data_root, f)) for s, f in SPLITS.items()}
    plan = [(s, a, min(len(data[s]), a + shard_size)) for s in SPLITS for a in range(0, len(data[s]), shard_size)]
    todo = [p for p in plan if not os.path.exists(shard_path(out_dir, p[0], p[1]))]
    total_chars = sum(min(len(t), max_len * 4) for s, a, b in todo for e in data[s][a:b] for t in claim_texts(e))
    log(f"{len(plan)} shards, {len(todo)} to do, ~{total_chars / 4 / 1e6:.1f}M tokens (after truncation to {max_len})")

    # ---- replicas: one per GPU, or one split over all GPUs when the model is too big ----
    n_gpu = torch.cuda.device_count()
    if n_gpu == 0:
        raise RuntimeError("no GPU visible: set Accelerator = GPU T4 x2")
    gpu_gib = min(torch.cuda.get_device_properties(i).total_memory for i in range(n_gpu)) / 2**30
    from huggingface_hub import HfApi
    try:
        size_gib = sum(s.size or 0 for s in HfApi().model_info(model, files_metadata=True).siblings
                       if s.rfilename.endswith(".safetensors")) / 2**30
    except Exception:
        size_gib = 16.0 if "8B" in model else (8.0 if "4B" in model else 1.2)
    if quant == "4bit":
        size_gib *= 0.37          # nf4 linear layers + fp16 embedding table (8B: ~15 -> ~5.5 GiB)
    fits_one = size_gib < gpu_gib - 2.5
    log(f"{n_gpu} GPU(s) x {gpu_gib:.1f} GiB | {model} {quant or ''} ~{size_gib:.1f} GiB weights | "
        f"{'one copy per GPU' if fits_one else 'single copy split over all GPUs'}")

    def build(dtype):
        if fits_one:
            return [Encoder(model, dtype, [i], max_len, gpu_mem, quant) for i in range(n_gpu)]
        return [Encoder(model, dtype, list(range(n_gpu)), max_len, gpu_mem, quant)]

    # ---- preflight on ~100 real texts ----
    sample = [t for e in data["test"][:12] for t in claim_texts(e)][:100]
    dtype = torch.float16
    encs = build(dtype)
    for attempt in range(2):
        t0 = time.time()
        _, ntok, nbad = embed_texts(encs[0], sample, token_budget, max_batch)
        torch.cuda.synchronize()
        tps = ntok / (time.time() - t0)
        log(f"preflight {str(dtype)}: {len(sample)} texts, {nbad} NaN rows, {tps:,.0f} padded tokens/s per replica, "
            f"GPU0 mem {torch.cuda.max_memory_allocated(0) / 2**30:.1f} GiB")
        if nbad == 0:
            break
        # fp32 compute: 4-bit weights stay 4-bit, so it always fits; fp16 weights must double
        if attempt == 0 and fits_one and (quant == "4bit" or 2 * size_gib < gpu_gib - 2.5):
            log("fp16 produced NaNs -> retrying in fp32")
            del encs
            torch.cuda.empty_cache()
            dtype = torch.float32
            encs = build(dtype)
        else:
            raise RuntimeError("model produces NaN embeddings in fp16 and fp32 does not fit: use "
                               "Qwen/Qwen3-Embedding-0.6B (fits in fp32) or a smaller max_len")
    eta_h = total_chars / 4 * 1.15 / (tps * len(encs)) / 3600   # +15% padding overhead
    log(f"ETA for the remaining shards: ~{eta_h:.1f} h with {len(encs)} replica(s) (time budget {time_budget_h} h)")
    if eta_h > time_budget_h:
        log("WARNING: will not finish in one session; finished shards are kept - re-run with resume_root next time")
    if preflight_only:
        return

    # ---- shards: each replica pulls from a shared queue ----
    q = queue.Queue()
    for p in todo:
        q.put(p)
    done = [0]
    errors = []
    lock = threading.Lock()

    def worker(r, enc):
        while True:
            if time.time() - t_start > time_budget_h * 3600:
                log(f"[replica {r}] time budget reached, stopping")
                return
            try:
                split, a, b = q.get_nowait()
            except queue.Empty:
                return
            try:
                t0 = time.time()
                items = data[split][a:b]
                texts, owner = [], []
                for i, e in enumerate(items):
                    ts = claim_texts(e)
                    texts += ts
                    owner += [i] * len(ts)
                E, ntok, nbad = embed_texts(enc, texts, token_budget, max_batch)
                owner = np.array(owner)
                out = {}
                for i, e in enumerate(items):
                    key = e.get("query_id", a + i) if split == "test" else a + i
                    out[f"id_{key}"] = E[owner == i]
                path = shard_path(out_dir, split, a)
                tmp = path[:-4] + ".tmp.npz"
                np.savez(tmp, **out)
                os.replace(tmp, path)
                with lock:
                    done[0] += 1
                    el = (time.time() - t_start) / 3600
                    log(f"[replica {r}] {split} {a}-{b}: {len(texts)} texts, {ntok / (time.time() - t0):,.0f} tok/s, "
                        f"NaN rows {nbad} | {done[0]}/{len(todo)} shards, elapsed {el:.2f} h")
            except Exception as ex:  # keep the other replica going
                errors.append(repr(ex))
                log(f"[replica {r}] ERROR on {split} {a}-{b}: {ex!r}")
                return

    threads = [threading.Thread(target=worker, args=(r, e), daemon=True) for r, e in enumerate(encs)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

    missing = [p for p in plan if not os.path.exists(shard_path(out_dir, p[0], p[1]))]
    if missing:
        log(f"{len(missing)}/{len(plan)} shards missing -> save this version's output as a dataset and "
            f"re-run with resume_root='/kaggle/input/<that dataset>'")
        return False
    for split in SPLITS:
        merged = {}
        for f in sorted(glob.glob(os.path.join(out_dir, "shards", f"{split}_*.npz"))):
            with np.load(f) as z:
                merged.update({k: z[k] for k in z.files})
        dst = os.path.join(out_dir, f"{split}_trace_emb_en.npz")
        np.savez(dst, **merged)
        log(f"merged {len(merged)} claims -> {dst}")
    return True


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="Qwen/Qwen3-Embedding-0.6B")
    ap.add_argument("--data-root", default="/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data")
    ap.add_argument("--out-dir", default="/kaggle/working/en_trace_emb")
    ap.add_argument("--resume-root", default=None)
    ap.add_argument("--shard-size", type=int, default=200)
    ap.add_argument("--max-len", type=int, default=512, help="token truncation per text")
    ap.add_argument("--token-budget", type=int, default=16384, help="max padded tokens per batch")
    ap.add_argument("--max-batch", type=int, default=64)
    ap.add_argument("--gpu-mem", default="13GiB", help="per-GPU weight cap when one copy is split over GPUs")
    ap.add_argument("--time-budget-h", type=float, default=9.0)
    ap.add_argument("--preflight-only", action="store_true")
    ap.add_argument("--quant", default=None, choices=[None, "4bit"], help="bitsandbytes nf4 weights")
    a = ap.parse_args()
    run(a.data_root, a.out_dir, a.model, a.max_len, a.shard_size, a.token_budget, a.max_batch,
        a.time_budget_h, a.resume_root, a.gpu_mem, a.preflight_only, a.quant)

In [ ]:
%%writefile /kaggle/working/rank_traces.py
"""
Recall@5-optimal ranking of LLM reasoning traces
CLEF CheckThat! 2026 Task 2 -- sub-problem B (trace ranking)

WHY THIS WORKS
--------------
The official scorer marks a trace relevant iff its verdict == gold label. So every trace
with the same verdict is interchangeable, and Recall@5 of a claim depends only on HOW MANY
traces of each verdict are in the top 5 (k_c), never on which ones:

    Recall@5 = k_y / n_y          (y = gold label, n_y = #traces with verdict y)

With p_c = P(gold = c | claim, traces), the expected Recall@5 of a top-5 allocation is

    E[R@5] = sum_c  p_c * k_c / n_c        subject to  sum_c k_c = 5,  0 <= k_c <= n_c

This is linear in k, so the exact optimum is greedy: rank verdict GROUPS by p_c / n_c and
list all traces of the best group first, then the next group, and so on. Rankers that
score traces one at a time (cosine, rerankers, reward models) cannot do this.

The problem therefore reduces to estimating a well-calibrated p_c per claim. Feature blocks
(logistic regression on their concatenation):
  counts : verdict-count vector                                    (always on)
  claim  : PCA of the claim embedding                              (needs npz)
  group  : per-verdict-group trace centroids, cohesion, claim sims (needs per-trace npz)
  knn    : similarity-weighted labels of the 10 nearest training claims
plus
  memory : a test claim whose exact text is in TRAIN+VAL takes that label (always on)
           (AR: 192/395, ES: 278/1135 scored test claims; labels agreed 100%)
  shift  : covariate-shift importance weights, train+val vs unlabeled test (Shimodaira 2000)

Blocks and the regularisation strength are chosen by 5-fold cross-validated Recall@5 on
TRAIN+VAL; the final model is refit on TRAIN+VAL and applied to TEST. Test labels are only
used for the final report. The ranking never uses the verdict classifier.

USAGE (CPU is enough)
    python rank_traces.py --lang all --data-root <data> --emb EN=<dir> --emb AR=<dir> --emb ES=<dir>
Outputs in --out-dir:
    {LANG}_test_predictions.json   scorer.py-compatible (query_id, Claim, Label, Verdict_BoN,
                                   BoN_Verdict_list, score_list, label_probs)
    {LANG}_results.csv             test Recall@k of all methods, all rows and deduped (scorer.py)
"""

import argparse
import csv
import glob
import json
import os
import re
import warnings
from collections import Counter

import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

CLASSES = ["true", "false", "conflicting"]
K = 5

# Data files (relative to --data-root; if missing, the basename is searched recursively)
# and embedding npz names (inside the folder given with --emb LANG=<dir>).
LANGS = {
    "EN": dict(
        train="English/train.json", val="English/validation.json", test="English/test.json",
        emb="{split}_trace_emb_en.npz",
    ),
    "AR": dict(
        train="Arabic/clef2026_gpt4_o_mini_train_arabic.json",
        val="Arabic/clef2026_gpt4_o_mini_val_arabic.json",
        test="Arabic/clef_arabic_test_gold_labels-final.json",
        emb="{split}_qwen3emb8b_ar_v1.npz",
    ),
    "ES": dict(
        train="Spanish/spanish_train.json", val="Spanish/spanish_val.json",
        test="Spanish/clef_spanish_test_final_with_gold_labels.json",
        emb="{split}_qwen3emb8b_ar_v1.npz",
    ),
}


def resolve(root, rel):
    p = os.path.join(root, rel)
    if os.path.exists(p):
        return p
    hits = glob.glob(os.path.join(root, "**", os.path.basename(rel)), recursive=True)
    parent = os.path.basename(os.path.dirname(rel)).lower()
    hits.sort(key=lambda h: parent not in h.lower())
    return hits[0] if hits else None


# ============================================================
# DATA
# ============================================================
def load_json(path):
    with open(path, encoding="utf-8") as f:
        s = f.read()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        # Local English train.json has one entry whose '"Reasoning_traces": [' key is missing
        return json.loads(re.sub(r'\{\s*\nx?(\s+)"(?![A-Za-z_]+"\s*:)', r'{\n        "Reasoning_traces": [\n\1"', s))


def norm(v):
    return str(v).strip().lower()


def claim_key(text):
    return re.sub(r"\s+", " ", str(text)).strip().lower()


class Data:
    """Arrays for one split. Embedding-derived fields stay None when no npz is available."""

    def __init__(self, items):
        self.items = items
        self.verdicts = [[norm(v) for v in e["Verdict_list"]] for e in items]
        self.gold = [norm(e.get("label", e.get("Label", ""))) for e in items]
        self.y = np.array([CLASSES.index(g) if g in CLASSES else -1 for g in self.gold])
        self.N = np.array([[sum(v == c for v in V) for c in CLASSES] for V in self.verdicts])
        self.qid = [e.get("query_id", e.get("query_index", i)) for i, e in enumerate(items)]
        self.keys = np.array([claim_key(e["claim"]) for e in items], dtype=object)
        self.claim = None       # (n, dim) L2-normalised claim embeddings
        self.claim_pca = None   # (n, d)
        self.group = None       # (n, 9 + 3d) per-verdict-group features
        self.rep = None         # per claim: per-trace representativeness (within-group tie-break)
        self.claim_cos = None   # per claim: per-trace claim cosine (old baseline)

    def take(self, idx):
        d = Data.__new__(Data)
        for k, v in self.__dict__.items():
            if isinstance(v, np.ndarray):
                setattr(d, k, v[idx])
            elif isinstance(v, list):
                setattr(d, k, [v[i] for i in idx])
            else:
                setattr(d, k, v)
        return d


def concat(a, b):
    d = Data.__new__(Data)
    for k, v in a.__dict__.items():
        w = getattr(b, k)
        if isinstance(v, np.ndarray) and isinstance(w, np.ndarray):
            setattr(d, k, np.concatenate([v, w]))
        elif isinstance(v, list) and isinstance(w, list):
            setattr(d, k, v + w)
        else:
            setattr(d, k, None)
    return d


def count_features(N):
    tot = N.sum(1, keepdims=True).clip(min=1)
    return np.hstack([N / tot, np.log1p(N), (N == 0).astype(float)])


# ============================================================
# EMBEDDINGS
# npz layout: key "id_{qid}" (test) or "id_{position}" (train/val) -> (1 + #non-empty traces, dim)
# row 0 = claim, rows 1.. = non-empty reasoning traces in original order.
# A claim-only npz (1 row per claim) is fine: the group block is then disabled.
# ============================================================
def npz_key(z, d, i):
    for k in (f"id_{d.qid[i]}", f"id_{i}"):
        if k in z.files:
            return k
    return None


def read_npz(path, d):
    """Streams one claim at a time and keeps only the claim vector + 3 group centroids per claim,
    so RAM stays small even for 8B float32 embeddings."""
    z = np.load(path)
    claims, cents, coh, rep, ccos, has_traces, missing = [], [], [], [], [], False, 0
    for i, e in enumerate(d.items):
        n_tr = len(d.verdicts[i])
        k = npz_key(z, d, i)
        if k is None:
            missing += 1
            claims.append(None)
            cents.append([None] * 3)
            coh.append([0.0] * 3)
            rep.append(np.zeros(n_tr))
            ccos.append(np.zeros(n_tr))
            continue
        a = z[k].astype(np.float32)
        a /= np.linalg.norm(a, axis=1, keepdims=True) + 1e-9
        claims.append(a[0])
        keep = [j for j, t in enumerate(e["Reasoning_traces"]) if str(t).strip()][: len(a) - 1]
        tr = a[1 : 1 + len(keep)]
        has_traces |= len(keep) > 0
        r, c = np.full(n_tr, -1.0), np.full(n_tr, -1.0)
        if keep:
            c[keep] = tr @ a[0]
        V = [d.verdicts[i][j] for j in keep]
        cc, hh = [], []
        for cls in CLASSES:
            g = [j for j, v in enumerate(V) if v == cls]
            if g:
                m = tr[g].mean(0)
                hh.append(float(np.linalg.norm(m)))          # ~ mean pairwise cosine in the group
                m = m / (np.linalg.norm(m) + 1e-9)
                r[[keep[j] for j in g]] = tr[g] @ m          # representativeness (tie-break only)
                cc.append(m)
            else:
                hh.append(0.0)
                cc.append(None)
        cents.append(cc)
        coh.append(hh)
        rep.append(r)
        ccos.append(c)
    z.close()
    dim = next(c for c in claims if c is not None).shape[0]
    d.claim = np.vstack([c if c is not None else np.zeros(dim, np.float32) for c in claims])
    d.rep, d.claim_cos = rep, ccos
    d.cents, d.coh, d.has_traces = cents, np.array(coh), has_traces
    if missing:
        print(f"    WARNING: {missing}/{len(d.items)} claims missing from {os.path.basename(path)}")


def finish_embedding_features(splits, dim):
    """PCA fitted on TRAIN+VAL claims and group centroids (unsupervised), applied to every split."""
    fit = []
    for s in ("train", "val"):
        fit.append(splits[s].claim)
        fit += [c[None] for cc in splits[s].cents[:3000] for c in cc if c is not None]
    pca = PCA(dim, random_state=0).fit(np.vstack(fit))
    zero = np.zeros(dim)
    for d in splits.values():
        d.claim_pca = pca.transform(d.claim)
        if d.has_traces:
            rows = []
            for i, cc in enumerate(d.cents):
                sims = [float(c @ d.claim[i]) if c is not None else 0.0 for c in cc]
                pair = [float(cc[a] @ cc[b]) if cc[a] is not None and cc[b] is not None else 0.0
                        for a, b in ((0, 1), (0, 2), (1, 2))]
                blocks = [pca.transform(c[None])[0] if c is not None else zero for c in cc]
                rows.append(np.concatenate([sims, d.coh[i], pair, *blocks]))
            d.group = np.array(rows, np.float32)
        del d.cents, d.coh


# ============================================================
# CLAIM MEMORY
# ============================================================
def exact_memory(bank, q):
    """Label distribution of bank claims with identical text."""
    idx = {}
    for j in np.where(bank.y >= 0)[0]:
        idx.setdefault(bank.keys[j], []).append(j)
    hit, dist = np.zeros(len(q.y), bool), np.zeros((len(q.y), len(CLASSES)))
    for i, k in enumerate(q.keys):
        js = idx.get(k)
        if js:
            hit[i] = True
            dist[i] = np.bincount(bank.y[js], minlength=len(CLASSES)) / len(js)
    return hit, dist


def knn_features(bank, q, same, k=10, chunk=2048):
    """Top-1 / mean cosine to the k nearest training claims and their similarity-weighted labels."""
    ok = np.where(bank.y >= 0)[0]
    Cb, yb = bank.claim[ok], bank.y[ok]
    pos = np.full(len(bank.y), -1)
    pos[ok] = np.arange(len(ok))
    F = np.zeros((len(q.y), 2 + len(CLASSES)))
    for s in range(0, len(q.y), chunk):
        S = q.claim[s : s + chunk] @ Cb.T
        if same:  # leave-one-out: a claim may not vote for itself
            rows = np.arange(s, min(s + chunk, len(q.y)))
            cols = pos[rows]
            S[(rows - s)[cols >= 0], cols[cols >= 0]] = -1
        top = np.argpartition(-S, k, axis=1)[:, :k]
        for r, t in enumerate(top):
            sim = S[r, t]
            w = np.maximum(sim, 0) ** 4
            F[s + r, 0], F[s + r, 1] = sim.max(), sim.mean()
            F[s + r, 2:] = np.bincount(yb[t], weights=w, minlength=len(CLASSES)) / max(w.sum(), 1e-9)
    return F


# ============================================================
# MODEL
# ============================================================
def design(bank, q, cfg, same):
    X = [count_features(q.N)]
    if cfg["claim"]:
        X.append(q.claim_pca)
    if cfg["group"]:
        X.append(q.group)
    if cfg["knn"]:
        X.append(knn_features(bank, q, same))
    return np.hstack(X)


def fit_predict(tr, te, cfg, weights=None):
    """Fits configuration `cfg` on `tr`; returns label probabilities for `te`."""
    ok = tr.y >= 0
    Xtr, Xte = design(tr, tr, cfg, same=True), design(tr, te, cfg, same=False)
    sc = StandardScaler().fit(Xtr[ok])
    m = LogisticRegression(C=cfg["C"], max_iter=5000)
    m.fit(sc.transform(Xtr[ok]), tr.y[ok], sample_weight=None if weights is None else weights[ok])
    P = np.zeros((len(te.y), len(CLASSES)))
    P[:, m.classes_] = m.predict_proba(sc.transform(Xte))
    hit, dist = exact_memory(tr, te)
    P[hit] = 0.9 * dist[hit] + 0.1 * P[hit]
    return P


def shift_weights(tv, te, clip):
    """Importance weights p(test|x)/p(train|x) from a domain classifier on counts (+ claim PCA)."""
    blocks = lambda d: np.hstack([count_features(d.N)] + ([d.claim_pca] if d.claim_pca is not None else []))
    X = np.vstack([blocks(tv), blocks(te)])
    dom = np.r_[np.zeros(len(tv.y)), np.ones(len(te.y))]
    sc = StandardScaler().fit(X)
    m = LogisticRegression(C=0.1, max_iter=5000).fit(sc.transform(X), dom)
    p = m.predict_proba(sc.transform(blocks(tv)))[:, 1]
    w = np.clip(p / (1 - p) * len(tv.y) / len(te.y), 0, clip)
    return w / w.mean()


# ============================================================
# RANKING + EVALUATION
# ============================================================
def rank_claim(p, verdicts, rep=None):
    """Trace order: verdict groups by p_c / n_c, within a group by representativeness."""
    n = Counter(verdicts)
    prio = {v: (p[CLASSES.index(v)] / n[v] if v in CLASSES else -1.0) for v in n}
    rep = np.zeros(len(verdicts)) if rep is None else rep
    return sorted(range(len(verdicts)), key=lambda i: (-prio[verdicts[i]], -rep[i], i))


def orders_from_probs(P, d):
    reps = d.rep or [None] * len(d.y)
    return [rank_claim(p, V, r) for p, V, r in zip(P, d.verdicts, reps)]


def recall_at_k(order, verdicts, gold, k):
    total = sum(v == gold for v in verdicts)
    return sum(verdicts[i] == gold for i in order[:k]) / total if total else 0.0


def evaluate(orders, d, dedupe):
    """dedupe=True mirrors given code/scorer.py (first row per claim text only)."""
    seen, r = set(), {k: [] for k in range(1, K + 1)}
    for i, o in enumerate(orders):
        c = d.items[i]["claim"]
        if dedupe and c in seen:
            continue
        seen.add(c)
        for k in r:
            r[k].append(recall_at_k(o, d.verdicts[i], d.gold[i], k))
    return {k: float(np.mean(v)) for k, v in r.items()}


def baseline_orders(d, kind, seed=0):
    rng = np.random.default_rng(seed)
    out = []
    for i, V in enumerate(d.verdicts):
        n = Counter(V)
        if kind == "random":
            out.append(list(rng.permutation(len(V))))
        elif kind == "majority_first":
            out.append(sorted(range(len(V)), key=lambda j: (-n[V[j]], j)))
        elif kind == "minority_first":
            out.append(sorted(range(len(V)), key=lambda j: (n[V[j]], j)))
        elif kind == "oracle":
            out.append(sorted(range(len(V)), key=lambda j: (V[j] != d.gold[i], j)))
        elif kind == "claim_cosine":
            out.append(list(np.argsort(-d.claim_cos[i], kind="stable")))
    return out


def cv_score(tv, cfg, folds):
    P = np.zeros((len(tv.y), len(CLASSES)))
    for a, b in folds:
        P[b] = fit_predict(tv.take(a), tv.take(b), cfg)
    return evaluate(orders_from_probs(P, tv), tv, dedupe=False)[K]


# ============================================================
# MAIN
# ============================================================
def run_lang(lang, args):
    spec = LANGS[lang]
    paths = {s: resolve(args.data_root, spec[s]) for s in ("train", "val", "test")}
    if not all(paths.values()):
        print(f"[{lang}] skipped, missing files: {paths}")
        return None
    print(f"\n================ {lang} ================", flush=True)
    S = {s: Data(load_json(p)) for s, p in paths.items()}
    for s in S:
        print(f"  {s:5s}: {len(S[s].y)} claims | labels {dict(Counter(S[s].gold))}")

    emb_dir = dict(e.split("=", 1) for e in args.emb).get(lang)
    use_emb = use_group = False
    if emb_dir and not args.no_emb:
        npz = {s: resolve(emb_dir, spec["emb"].format(split=s)) for s in S}
        if all(npz.values()):
            for s in S:
                print(f"  reading {npz[s]}", flush=True)
                read_npz(npz[s], S[s])
            use_group = all(S[s].has_traces for s in S)
            finish_embedding_features(S, args.pca_dim)
            use_emb = True
        else:
            print(f"  embeddings not found: {npz}")
    print("  feature blocks: counts, memory" + (", claim, knn" if use_emb else "") + (", group" if use_group else ""))

    tv = concat(S["train"], S["val"])
    te = S["test"]

    # ---- block / C selection by 5-fold CV Recall@5 on TRAIN+VAL ----
    cands = [dict(claim=False, group=False, knn=False, C=1.0)]
    if use_emb:
        for claim in (False, True):
            for group in ((False, True) if use_group else (False,)):
                for knn in (False, True):
                    if claim or group or knn:
                        cands += [dict(claim=claim, group=group, knn=knn, C=C) for C in (0.01, 0.1)]
    folds = list(StratifiedKFold(5, shuffle=True, random_state=0).split(np.zeros(len(tv.y)), np.maximum(tv.y, 0)))
    scores = []
    for cfg in cands:
        scores.append(cv_score(tv, cfg, folds))
        print(f"    CV R@5 {scores[-1]:.4f}  {cfg}", flush=True)
    top = max(scores)
    # Simplest candidate (fewest blocks) within --min-gain of the best CV score
    size = lambda c: c["claim"] + c["group"] + c["knn"]
    best = min((c for c, s in zip(cands, scores) if s >= top - args.min_gain), key=size)
    print(f"  selected {best} (CV {scores[cands.index(best)]:.4f}, best {top:.4f})")

    # ---- final fit on TRAIN+VAL -> TEST ----
    P = fit_predict(tv, te, best)
    methods = {m: baseline_orders(te, m) for m in ("random", "majority_first", "minority_first", "oracle")}
    if use_emb:
        methods["claim_cosine (old)"] = baseline_orders(te, "claim_cosine")
    methods["p/n counts"] = orders_from_probs(fit_predict(tv, te, cands[0]), te)
    methods["p/n selected"] = orders_from_probs(P, te)
    final = "p/n selected"
    if args.shift_clip > 0:
        P = fit_predict(tv, te, best, weights=shift_weights(tv, te, args.shift_clip))
        final = "p/n selected + shift"
        methods[final] = orders_from_probs(P, te)

    results = {m: (evaluate(o, te, dedupe=False), evaluate(o, te, dedupe=True)) for m, o in methods.items()}
    write_predictions(os.path.join(args.out_dir, f"{lang}_test_predictions.json"), te, P, methods[final])

    out_csv = os.path.join(args.out_dir, f"{lang}_results.csv")
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["method"] + [f"R@{k} allrows" for k in range(1, K + 1)] + ["R@5 dedup (scorer.py)"])
        for m, (a, d) in results.items():
            w.writerow([m] + [round(a[k], 4) for k in range(1, K + 1)] + [round(d[K], 4)])
    print(f"\n  TEST {'method':24s} R@5 all rows   R@5 dedup (scorer.py)")
    for m, (a, d) in results.items():
        print(f"       {m:24s} {a[K]:.4f}         {d[K]:.4f}")
    print(f"  final ranking = '{final}' -> {out_csv}")
    return {m: (a[K], d[K]) for m, (a, d) in results.items()}


def write_predictions(path, d, P, orders):
    out = []
    for i, e in enumerate(d.items):
        score = [0.0] * len(d.verdicts[i])
        for r, j in enumerate(orders[i]):
            score[j] = float(len(orders[i]) - r)
        out.append({
            "query_id": d.qid[i],
            "Claim": e["claim"],
            "Label": e.get("label", e.get("Label")),
            # Most probable label, NOT the top-ranked trace's verdict (the p/n rule often puts a
            # small minority group first on purpose).
            "Verdict_BoN": CLASSES[int(np.argmax(P[i]))].capitalize(),
            "BoN_Verdict_list": e["Verdict_list"],
            "score_list": score,
            "label_probs": {c.capitalize(): round(float(p), 5) for c, p in zip(CLASSES, P[i])},
        })
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--lang", default="all", help="EN | AR | ES | all")
    ap.add_argument("--data-root", default="/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data")
    ap.add_argument("--emb", action="append", default=[], help="LANG=<folder with that language's npz files>")
    ap.add_argument("--no-emb", action="store_true", help="counts + memory only")
    ap.add_argument("--pca-dim", type=int, default=32)
    ap.add_argument("--min-gain", type=float, default=0.001, help="CV R@5 margin an extra feature block must win by")
    ap.add_argument("--shift-clip", type=float, default=5.0, help="covariate-shift weight clip; 0 disables")
    ap.add_argument("--out-dir", default="/kaggle/working/trace_ranking")
    args = ap.parse_args()
    os.makedirs(args.out_dir, exist_ok=True)
    summary = {}
    for lang in (LANGS if args.lang == "all" else [args.lang.upper()]):
        r = run_lang(lang, args)
        if r:
            summary[lang] = r
    print("\n==== SUMMARY: test Recall@5 (all rows / dedup) ====")
    for lang, r in summary.items():
        print(f"  {lang}: " + " | ".join(f"{m} {a:.4f}/{d:.4f}" for m, (a, d) in r.items()
                                        if m in ("oracle", "p/n counts", "p/n selected", "p/n selected + shift")))

## 4. Stage A — rank all languages without new English embeddings (CPU, ~20 min)

In [ ]:
cmd = [sys.executable, "-u", f"{WORK}/rank_traces.py", "--lang", "all", "--data-root", DATA_ROOT, "--out-dir", f"{WORK}/results_A"]
if AR_EMB_DIR: cmd += ["--emb", f"AR={AR_EMB_DIR}"]
if ES_EMB_DIR: cmd += ["--emb", f"ES={ES_EMB_DIR}"]
run_live(cmd)

## 5. Stage B — English per-trace embeddings (GPU)
The preflight prints tokens/s, NaN rows, GPU memory and the ETA before the long run starts.
Shards (200 claims) are saved as they finish. If the run stops early, save the version, attach its
output as a dataset, set `RESUME_ROOT` to it and run again: finished shards are skipped.

In [ ]:
sys.path.insert(0, WORK)
import importlib, embed_traces
importlib.reload(embed_traces)
EN_DONE = embed_traces.run(data_root=DATA_ROOT, out_dir=EN_EMB_DIR, model=EN_EMB_MODEL, max_len=EN_MAX_LEN,
                           time_budget_h=TIME_BUDGET_H, resume_root=RESUME_ROOT, quant=EN_QUANT)
print("English embeddings complete:", EN_DONE)

## 6. Stage C — rank English with its embeddings (CPU, ~10 min)

In [ ]:
if EN_DONE:
    run_live([sys.executable, "-u", f"{WORK}/rank_traces.py", "--lang", "EN", "--data-root", DATA_ROOT,
              "--emb", f"EN={EN_EMB_DIR}", "--out-dir", f"{WORK}/results_C"])
else:
    print("English embeddings incomplete -> skipped; Stage A results are still valid")

## 7. Results
*all rows* = every test row scored; *dedup* = official `scorer.py` (first row per claim text).

In [ ]:
import pandas as pd
for stage in ("results_A", "results_C"):
    for lang in ("EN", "AR", "ES"):
        f = f"{WORK}/{stage}/{lang}_results.csv"
        if os.path.exists(f):
            print(f"\n===== {stage} | {lang} =====")
            display(pd.read_csv(f).set_index("method"))

## 8. Cross-check with the official `scorer.py` logic

In [ ]:
import json
import numpy as np

def official_recall5(path):
    data = json.load(open(path, encoding="utf-8"))
    seen, rs = set(), []
    for e in data:
        if e["Claim"] in seen:
            continue
        seen.add(e["Claim"])
        label = e["Label"].lower()
        verdicts = [v.lower() for v in e["BoN_Verdict_list"]]
        ranked = sorted(range(len(e["score_list"])), key=lambda i: e["score_list"][i], reverse=True)
        total = sum(v == label for v in verdicts)
        rs.append(sum(verdicts[i] == label for i in ranked[:5]) / total if total else 0.0)
    return np.mean(rs), len(rs)

for f in sorted(glob.glob(f"{WORK}/results_*/*_test_predictions.json")):
    r, n = official_recall5(f)
    print(f"{f.split('/')[-2]} {os.path.basename(f)[:2]}: official scorer Recall@5 = {r:.4f} on {n} unique claims")